In [ ]:
# Allow importing from src
import sys
sys.path.insert(0, '../src/')

# Fix for draw_geometries crashing on Wayland
import os
os.environ["XDG_SESSION_TYPE"] = "x11"

In [ ]:
import open3d as o3d
from pathlib import Path
import numpy as np
import torch
from matplotlib import pyplot as plt
import cv2
import open3d as o3d
from PIL import Image
from torchvision.utils import make_grid
import re
import json

# DTU

In [ ]:
print("Available scans:")
sc_list = [int(p.stem[4:]) for p in Path(f'../data/DTU').iterdir() if p.is_dir()]
sc_list.sort()
print(", ".join([str(s) for s in sc_list]))

## Images

In [ ]:
SCAN_NUM = 24
MASKED = True

scan = Path(f"../data/DTU/scan{SCAN_NUM}").resolve()
imgs = (scan / "image").glob("[0-9]*.png")
masks = (scan / "mask").glob("[0-9]*.png")

ls = []
for img, mask in zip(sorted(imgs), sorted(masks)):
    image = np.asarray(Image.open(img))
    if MASKED:
        alpha = np.asarray(Image.open(mask))
        image = np.concat([image, alpha.mean(-1, keepdims=True)], axis=-1)
        
    ls.append(image / 255)

imgs = np.stack(ls, 0)

fig, ax = plt.subplots(1, 1, figsize=(25, 25))
ax.imshow(make_grid(torch.tensor(imgs).permute(0, 3, 1, 2), 5, padding=10).permute(1, 2, 0))
ax.axis('off')
pass

## C2W and Intrinsic matrices

IDR provides scale matrices to normalize the scene to approx the unit sphere, very good for INGP as the distro's
implementation expects objects to fit into the [-1, 1] bounding box.

In [ ]:
arr = np.load(scan / "cameras.npz")
# arr['scale_mat_0'], arr['scale_mat_inv_0'], arr['world_mat_0'], arr['world_mat_inv_0'], arr['camera_mat_0'], arr['camera_mat_inv_0']

world_mat = arr['world_mat_0']
scale_mat = arr['scale_mat_0']

# This function is borrowed from IDR: https://github.com/lioryariv/idr
def load_K_Rt_from_P(P):
    out = cv2.decomposeProjectionMatrix(P)
    K = out[0]
    R = out[1]
    t = out[2]

    K = K / K[2, 2]
    intrinsics = np.eye(4)
    intrinsics[:3, :3] = K

    pose = np.eye(4, dtype=np.float32)
    pose[:3, :3] = R.transpose()
    pose[:3, 3] = (t[:3] / t[3])[:, 0]

    return intrinsics, pose

pts = []
x1 = []
y1 = []
directions = []
for i in range(len(arr) // 6):
    P = arr[f'world_mat_{i}'] @ arr[f'scale_mat_{i}']
    P = P[:3, :4]
    intrinsics, pose = load_K_Rt_from_P(P)
    # transforming z-forward, y-down to z-backward, y-up
    pose[:3, 1] *= -1
    pose[:3, 2] *= -1
    pts.append(pose[:3, -1])

    direction = np.array((0, 0, -1), np.float32) @ pose[:3, :3].T
    direction /= np.linalg.norm(direction)

    x1.append(np.array([1,0,0]) @ pose[:3,:3].T * 0.1 + pose[:3, -1])
    y1.append(np.array([0,1,0]) @ pose[:3,:3].T * 0.1 + pose[:3, -1])

    directions.append(direction)

pts

pose_locs = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(pts))
pose_locs.normals = o3d.utility.Vector3dVector(directions)
pose_locs.paint_uniform_color([0.2, 0.7, 0.2])

x1_locs = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(x1))
x1_locs.paint_uniform_color([0.7, 0.2, 0.2])

y1_locs = o3d.geometry.PointCloud(o3d.utility.Vector3dVector(y1))
y1_locs.paint_uniform_color([0.2, 0.7, 0.7])

unit_sphere = o3d.geometry.TriangleMesh.create_sphere()
unit_sphere = o3d.geometry.LineSet.create_from_triangle_mesh(unit_sphere)

axis = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.2)

o3d.visualization.draw_geometries([
   pose_locs, unit_sphere, axis, x1_locs, y1_locs
], point_show_normal=True)

In [ ]:
arr['scale_mat_0'], arr['scale_mat_1'], arr['scale_mat_2']

## Scaling the STL

In [ ]:
point_cloud = o3d.io.read_point_cloud(scan / f"stl{SCAN_NUM:03d}_total.ply")
point_cloud.transform(arr['scale_mat_inv_0'])
point_cloud = point_cloud.voxel_down_sample(0.005)

unit_box = o3d.geometry.TriangleMesh.create_box(2.0, 2.0, 2.0)
unit_box.translate([-1, -1, -1])
unit_box = o3d.geometry.LineSet.create_from_triangle_mesh(unit_box)

o3d.visualization.draw_geometries([
    point_cloud, unit_box
])

# NeRF Synthetic

In [ ]:
print("Available scenes:")
sc_list = [p.stem for p in Path(f'../data/NeSy').iterdir() if p.is_dir()]
sc_list.sort()
print(", ".join([str(s) for s in sc_list]))

## Images, C2Ws and Intrinsics

In [ ]:
SCENE_NAME = "lego"

scene = Path(f"../data/NeSy/{SCENE_NAME}").resolve()
with open(scene / "transforms_train.json", "r") as f:
    transforms = json.load(f)

imgs, c2ws = [], []
for frame in transforms["frames"]:
    image = torch.from_numpy(np.asarray(Image.open(scene / f"{frame["file_path"]}.png"), dtype=np.float32))
    imgs.append(image / 255)
    c2ws.append(torch.tensor(frame["transform_matrix"], dtype=torch.float32))

imgs, c2ws = torch.stack(imgs, 0), torch.stack(c2ws, 0)

angle = torch.tensor(transforms["camera_angle_x"], dtype=torch.float32)
height, width = imgs.shape[1], imgs.shape[2]
focal = (width / 2.0) / torch.tan(angle / 2.0)
intrinsics = torch.tensor([
    [focal, 0.0, width / 2.0],
    [0.0, focal, height / 2.0],
    [0.0, 0.0, 1.0],
], dtype=torch.float32).unsqueeze(0).expand(imgs.shape[0], -1, -1)

fig, ax = plt.subplots(1, 1, figsize=(25, 50))
ax.imshow(make_grid(imgs.permute(0, 3, 1, 2), 5, padding=10).permute(1, 2, 0))
ax.axis('off')
pass